# Notebook 2 — Interaction Features

**Difficulty:** ⭐⭐⭐ | **Estimated time:** 50 min

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain *when* and *why* interaction features are needed.
2. Implement pairwise products and polynomial features **from scratch**.
3. Use `sklearn.preprocessing.PolynomialFeatures` intelligently and verify it against your scratch implementation.
4. Avoid the curse of dimensionality by **screening interactions** with mutual information.

---

**Week 12 Theme — Airbnb Listing Price Prediction**

We continue with the synthetic Airbnb dataset. This notebook focuses on capturing **non-linear synergies** between features — things that are worth more (or less) than the sum of their parts.

## The Synergy Analogy

Imagine two Airbnb listings:

- **Listing A**: 3 bedrooms, mediocre location.
- **Listing B**: 3 bedrooms, prime city-centre location.

The extra bedrooms add value in *both* listings — but they add **much more value** in the prime location because guests are already willing to pay top dollar, and extra space pushes the price even higher.

A linear model adds features independently:
$$\hat{\text{price}} = w_1 \cdot \text{bedrooms} + w_2 \cdot \text{location\_score} + \ldots$$

This model gives the **same extra value per bedroom** regardless of location. It misses the synergy.

An **interaction feature** captures it explicitly:
$$\text{bedrooms} \times \text{location\_score}$$

Now the model can assign a weight to the *combined* effect — the synergy is visible as a single number.

## Why Does This Matter for ML?

- **Linear models cannot represent multiplicative effects** without explicit interaction features. If the true relationship involves $x_1 \times x_2$, a linear model will underfit unless you add that product as a column.
- **Some tree models learn interactions automatically** (via multi-level splits), but shallow trees and gradient boosting with low depth miss deep interactions.
- **Interactions are the most common way to capture non-linearity in structured tabular data** without switching to a more complex model.
- In Kaggle competitions, manually engineered interaction features based on domain knowledge regularly outperform brute-force polynomial expansion.
- **Interpretability bonus**: an interaction feature like `review_score × num_reviews` is easy to explain — "we trust a high score more when it comes from many reviews".

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_absolute_error

np.random.seed(42)                    # global seed for reproducibility
sns.set_theme(style='whitegrid')      # consistent plot style

print('Libraries loaded successfully.')

## Mathematical View — Why Linear Models Need Explicit Interactions

A linear model with two features $x_1$ and $x_2$ fits:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2$$

The response surface is a **flat plane**. No matter what weights you choose, you cannot represent:

$$y = 2 \cdot x_1 \cdot x_2 + \varepsilon$$

because that is a **saddle surface** (hyperbolic paraboloid), not a plane.

**The fix**: add $x_3 = x_1 \times x_2$ as an explicit feature. Now the model becomes:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + w_3 (x_1 x_2)$$

With $w_3 = 2$ and $w_1 = w_2 = 0$, the model recovers the true relationship exactly.

The model is still **linear in its parameters** ($w_0, w_1, w_2, w_3$) — just non-linear in its original inputs. This is the key insight: polynomial and interaction features let linear models learn non-linear functions.

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

N = 1000

# ── Synthetic data where the true relationship is a product ───────────────
bedrooms   = np.random.randint(1, 7, N).astype(float)
is_central = np.random.randint(0, 2, N).astype(float)  # 1 = city centre, 0 = suburb

# True data-generating process: only the PRODUCT matters
y = 2.0 * bedrooms * is_central + np.random.normal(0, 1, N)

# ── Feature matrices ──────────────────────────────────────────────────────
X_no_int   = np.column_stack([bedrooms, is_central])                          # 2 features
X_with_int = np.column_stack([bedrooms, is_central, bedrooms * is_central])   # 3 features

# ── Fit LinearRegression on both ──────────────────────────────────────────
from sklearn.linear_model import LinearRegression

for label, X in [('Without interaction', X_no_int), ('With interaction feature', X_with_int)]:
    lr = LinearRegression()
    cv_r2 = cross_val_score(lr, X, y, cv=5, scoring='r2').mean()
    lr.fit(X, y)  # refit on full data to inspect coefficients
    coef_str = ', '.join(f'{c:+.3f}' for c in lr.coef_)
    print(f'{label:30s}: CV R² = {cv_r2:.4f}  | coefs = [{coef_str}]')

print()
print('Expected: without interaction, R² ≈ 0 (pure additive model cannot fit a product).')
print('With interaction, R² ≈ 1.0 and the interaction coefficient ≈ +2.0.')

# ── Visualise residuals ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (label, X) in zip(axes, [('No interaction', X_no_int), ('With interaction', X_with_int)]):
    lr = LinearRegression().fit(X, y)
    residuals = y - lr.predict(X)
    ax.scatter(lr.predict(X), residuals, alpha=0.3, s=12)
    ax.axhline(0, color='red', linewidth=1.5)             # zero-residual reference line
    ax.set_xlabel('Fitted values')
    ax.set_ylabel('Residuals')
    ax.set_title(f'{label}\n(structured residuals = missed pattern)')

plt.suptitle('Residual Plots: Interaction Feature Eliminates Systematic Error', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Pairwise Interaction Features — The Explosion Problem

For $p$ features, the number of distinct pairwise products (including $x_i^2$ terms) is:

$$\text{interactions} = \frac{p(p+1)}{2}$$

| Features $p$ | Pairwise interactions | Total features with originals |
|---|---|---|
| 5 | 15 | 20 |
| 10 | 55 | 65 |
| 20 | 210 | 230 |
| 50 | 1 275 | 1 325 |
| 100 | 5 050 | 5 150 |

If you have 100 features and 1 000 samples, brute-force expansion gives you 5 000+ features — you are in severe underdetermined territory. **Always screen interactions** before adding them all.

The cross-only count (excluding $x_i^2$) is $p(p-1)/2$. For $p=10$ that is 45 pairs — still large, but more manageable.

In [ ]:
np.random.seed(42)

# ── Rebuild the Airbnb dataset ────────────────────────────────────────────
N = 2000

bedrooms              = np.random.randint(1, 7, N).astype(float)
accommodates          = bedrooms * 2 + np.random.randint(0, 3, N)
bathrooms             = np.round(bedrooms * 0.6 + np.random.uniform(0, 0.5, N), 1)
review_score          = np.clip(np.random.normal(4.5, 0.4, N), 1, 5)
host_experience_days  = np.random.exponential(500, N).astype(int)
is_superhost          = (review_score > 4.7).astype(int)
distance_to_centre    = np.abs(np.random.normal(3.5, 2.0, N))
num_reviews           = np.random.negative_binomial(n=5, p=0.05, size=N)

price = (
    40
    + 25  * bedrooms
    + 15  * bathrooms
    + 18  * review_score
    - 5   * distance_to_centre
    + 0.02 * host_experience_days
    + 20  * is_superhost
    + np.random.normal(0, 15, N)
)
price = np.clip(price, 20, 600)

df = pd.DataFrame({
    'bedrooms':             bedrooms,
    'bathrooms':            bathrooms,
    'accommodates':         accommodates,
    'review_score':         review_score,
    'distance_to_centre':   distance_to_centre,
    'host_experience_days': host_experience_days,
    'is_superhost':         is_superhost,
    'num_reviews':          num_reviews,
    'price':                price
})

print(f'Dataset shape: {df.shape}')
df.head(3)

In [ ]:
# ── From-scratch pairwise interaction function ─────────────────────────────
def pairwise_interactions(X, feature_names=None):
    """
    Compute all pairwise products X[:, i] * X[:, j] for i <= j.
    This includes squared terms (i == j) as well as cross-terms (i != j).

    Parameters
    ----------
    X : ndarray of shape (n_samples, n_features)
    feature_names : list of str, optional

    Returns
    -------
    interactions : ndarray of shape (n_samples, n_features*(n_features+1)//2)
    names        : list of str (only if feature_names provided)
    """
    n, p = X.shape
    interactions = []   # will collect one column per interaction
    names = []

    for i in range(p):                  # iterate over all feature pairs
        for j in range(i, p):           # j >= i avoids duplicate pairs
            interactions.append(X[:, i] * X[:, j])   # element-wise product
            if feature_names is not None:
                if i == j:
                    names.append(f'{feature_names[i]}^2')          # squared term
                else:
                    names.append(f'{feature_names[i]}x{feature_names[j]}')  # cross-term

    interaction_matrix = np.column_stack(interactions)  # shape: (n, p*(p+1)/2)
    return interaction_matrix, names


# ── Apply to the 6 numeric Airbnb features ────────────────────────────────
numeric_feature_names = [
    'bedrooms', 'bathrooms', 'accommodates',
    'review_score', 'distance_to_centre', 'host_experience_days'
]
X_numeric = df[numeric_feature_names].values.astype(float)
y         = df['price'].values

X_interactions, interaction_names = pairwise_interactions(X_numeric, numeric_feature_names)

p = len(numeric_feature_names)
expected_n_interactions = p * (p + 1) // 2   # formula for including squared terms
print(f'Features: {p}')
print(f'Expected interactions (including squared): {expected_n_interactions}')
print(f'Actual interactions produced:              {X_interactions.shape[1]}')

# ── Show correlations of interactions with price ──────────────────────────
corr_with_price = pd.Series(
    [np.corrcoef(X_interactions[:, i], y)[0, 1] for i in range(X_interactions.shape[1])],
    index=interaction_names
).sort_values(key=abs, ascending=False)

print('\nTop 10 interactions by |correlation| with price:')
print(corr_with_price.head(10).to_string())

In [ ]:
np.random.seed(42)

# ── Verify scratch implementation against sklearn PolynomialFeatures ──────
# sklearn's PolynomialFeatures(degree=2, include_bias=False) generates:
#   - original features (degree 1)
#   - all pairwise products including squared terms (degree 2)
# Total: p + p*(p+1)/2 columns

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly_sklearn = poly.fit_transform(X_numeric)   # sklearn version

# Our scratch version gives ONLY the interaction/squared terms (not the originals)
# So total scratch columns = p*(p+1)/2
# sklearn total columns = p + p*(p+1)/2

n_original          = p                            # sklearn's degree-1 terms
n_interactions_poly = p * (p + 1) // 2            # sklearn's degree-2 terms
n_sklearn_total     = n_original + n_interactions_poly

print(f'sklearn PolynomialFeatures total columns : {X_poly_sklearn.shape[1]}')
print(f'Expected (p + p*(p+1)/2)                : {n_sklearn_total}')
print(f'Our scratch interaction columns          : {X_interactions.shape[1]}')
print(f'Expected scratch (p*(p+1)/2)             : {n_interactions_poly}')

# Verify the interaction columns of sklearn match our scratch implementation
# sklearn places original features in the first p columns, then degree-2 terms
X_poly_degree2_only = X_poly_sklearn[:, p:]       # drop the original features

max_diff = np.max(np.abs(X_poly_degree2_only - X_interactions))
print(f'\nMax absolute difference (scratch vs sklearn degree-2 terms): {max_diff:.2e}')
assert max_diff < 1e-8, 'Mismatch detected! Check column ordering.'
print('Assertion passed: scratch implementation matches sklearn exactly.')

## Polynomial Features — Adding Squared Terms

Interactions ($x_i \times x_j$ for $i \neq j$) capture synergies between *different* features.

**Squared terms** ($x_i^2$) capture **curvature** — U-shaped or inverted-U-shaped effects of a single feature on the target.

**Airbnb example:** `review_score` might have a non-linear effect:
- Very low scores (1–2) hurt bookings drastically → low price
- Average scores (3–4) have modest effect
- Very high scores (4.8–5.0) command a significant premium

This is a **J-curve** (accelerating positive effect), well approximated by adding `review_score²`.

More generally, for polynomial regression of degree $d$ with $p$ features, the total number of features is:

$$\binom{p + d}{d} = \frac{(p+d)!}{d! \cdot p!}$$

For $p=20$, $d=3$: that is **1771 features** from just 20 inputs. Always regularise.

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Demonstrate U-shaped / J-curve effect of review_score on price ────────
# We'll add a quadratic signal to make the curvature visible
review_range = np.linspace(1, 5, 300)
# Hypothetical non-linear pricing: massive penalty for low scores, premium for very high
price_curve = 80 + 10 * (review_range - 3)**2 * np.sign(review_range - 3)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: true curved relationship
axes[0].plot(review_range, price_curve, 'b-', linewidth=2.5)
axes[0].set_xlabel('review_score')
axes[0].set_ylabel('Expected price (USD/night)')
axes[0].set_title('J-curve: review_score effect on price\n(accelerating at both ends)')

# Right: compare linear vs quadratic fit on synthetic curved data
# Add synthetic quadratic signal to our Airbnb dataset temporarily
review_sq_signal = 5 * (df['review_score'] - 3)**2 * np.sign(df['review_score'] - 3)
y_curved = price + review_sq_signal     # augment price with quadratic review effect

x_plot  = df['review_score'].values.reshape(-1, 1)
x_quad  = np.column_stack([df['review_score'], df['review_score']**2])  # degree-2

from sklearn.linear_model import LinearRegression

lr_linear = LinearRegression().fit(x_plot, y_curved)
lr_quad   = LinearRegression().fit(x_quad, y_curved)

review_sorted = np.sort(df['review_score'].values)
axes[1].scatter(df['review_score'], y_curved, alpha=0.15, s=8, color='grey', label='Data')
axes[1].plot(review_sorted, lr_linear.predict(review_sorted.reshape(-1,1)),
             'r-', linewidth=2.5, label=f'Linear  R²={lr_linear.score(x_plot, y_curved):.3f}')
axes[1].plot(review_sorted, lr_quad.predict(np.column_stack([review_sorted, review_sorted**2])),
             'g-', linewidth=2.5, label=f'Quadratic R²={lr_quad.score(x_quad, y_curved):.3f}')
axes[1].set_xlabel('review_score')
axes[1].set_ylabel('Price (USD/night)')
axes[1].set_title('Linear vs Quadratic fit\nSquared term captures curvature')
axes[1].legend(fontsize=9)

plt.suptitle('Why Squared Terms Matter: Capturing Non-Linear Feature Effects', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ── Full polynomial expansion on Airbnb dataset ───────────────────────────
# PolynomialFeatures(degree=2) gives: originals + squared + cross-products

poly_full = PolynomialFeatures(degree=2, include_bias=False)
X_poly_full = poly_full.fit_transform(X_numeric)   # shape: (2000, p + p*(p+1)/2)

p = X_numeric.shape[1]
expected_cols = p + p * (p + 1) // 2
print(f'Original features   : {p}')
print(f'After degree-2 poly : {X_poly_full.shape[1]}  (expected {expected_cols})')

# ── Cross-validated Ridge comparison: before vs after polynomial expansion ─
y = df['price'].values

configs = [
    ('Raw features (6 cols)',         X_numeric),
    ('Degree-2 polynomial expansion', X_poly_full)
]

for label, X in configs:
    pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=10.0))])
    cv_r2 = cross_val_score(pipe, X, y, cv=5, scoring='r2')
    print(f'{label:40s}: CV R² = {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')

print('\nPolynomial expansion + Ridge regularisation often improves CV R²')
print('because it lets the linear model learn curvature and synergies.')

## Screening Interactions with Mutual Information

**Problem**: $p=20$ features gives 210 pairwise interactions. $p=50$ gives 1225. We cannot afford to add all of them, and regularisation can only do so much.

**Solution**: rank interactions by how much information they share with the target, then keep only the top $k$.

**Mutual information (MI)** measures the general statistical dependence between a feature and the target — it catches non-linear relationships that Pearson correlation misses:

$$I(X; Y) = \sum_{x,y} p(x,y) \log \frac{p(x,y)}{p(x)p(y)}$$

- MI = 0 → feature and target are statistically independent
- MI > 0 → knowing the feature reduces uncertainty about the target

**Practical screening workflow**:
1. Generate all pairwise interactions.
2. Compute MI between each interaction and the target.
3. Keep the top $k$ (e.g. top 10 or top 20).
4. Fit a regularised model with original features + selected interactions.
5. Compare CV score against baseline.

**Strategies for avoiding the explosion problem**:
- **(a) Domain-knowledge selection** — only consider interactions that make business sense.
- **(b) Regularisation** — add all interactions and let Lasso/Ridge zero out the irrelevant ones.
- **(c) MI screening** — data-driven ranking, model-agnostic, fast.

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Generate all pairwise cross-terms (no squared terms for clarity) ──────
# cross-only: i < j  → p*(p-1)/2 terms
p = X_numeric.shape[1]
cross_interactions = []
cross_names        = []

for i in range(p):
    for j in range(i + 1, p):                   # strictly i < j: cross-terms only
        cross_interactions.append(X_numeric[:, i] * X_numeric[:, j])
        cross_names.append(
            f'{numeric_feature_names[i]}x{numeric_feature_names[j]}'
        )

X_cross = np.column_stack(cross_interactions)    # shape: (2000, p*(p-1)/2)
print(f'Cross-interaction features generated: {X_cross.shape[1]}  '
      f'(expected {p*(p-1)//2})')

# ── Compute mutual information for each interaction ────────────────────────
mi_scores = mutual_info_regression(
    X_cross, y,
    discrete_features=False,  # all interactions are continuous
    random_state=42
)

mi_series = pd.Series(mi_scores, index=cross_names).sort_values(ascending=False)

# ── Bar chart of top-15 interactions ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
top15 = mi_series.head(15)
ax.barh(top15.index[::-1], top15.values[::-1], color='#4472c4')
ax.set_xlabel('Mutual Information with price', fontsize=11)
ax.set_title('Top-15 Pairwise Interactions Ranked by Mutual Information', fontsize=13)
plt.tight_layout()
plt.show()

print('\nTop 10 interactions by mutual information:')
print(mi_series.head(10).to_string())

In [ ]:
np.random.seed(42)

# ── Compare: (a) all interactions, (b) top-k MI-selected interactions ─────
top_k_values = [5, 10, 15, len(cross_names)]   # number of interactions to keep

print('Ridge regression CV R² — original + N interactions:')
print(f'{"N interactions":>25} | CV R²')
print('-' * 38)

# Baseline: original features only
pipe_base = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=10.0))])
r2_base = cross_val_score(pipe_base, X_numeric, y, cv=5, scoring='r2').mean()
print(f'{"0 (baseline)":>25} | {r2_base:.4f}')

# Sort interaction matrix columns by MI score (highest MI first)
mi_rank_idx = np.argsort(mi_scores)[::-1]          # descending MI order

for k in top_k_values:
    top_k_cols = mi_rank_idx[:k]                    # indices of top-k interactions
    X_top_k    = X_cross[:, top_k_cols]             # select those columns
    X_augmented = np.hstack([X_numeric, X_top_k])  # original + top-k interactions

    pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=10.0))])
    r2   = cross_val_score(pipe, X_augmented, y, cv=5, scoring='r2').mean()
    label = f'{k} (all)' if k == len(cross_names) else str(k)
    print(f'{label:>25} | {r2:.4f}')

print('\nObservation: selective top-k MI interactions often match or beat all interactions,')
print('especially with limited training data, because noise interactions add variance.')

## Multiplicative vs. Additive Interactions

A common confusion: what about *adding* features, e.g. $(x_1 + x_2)$?

A linear model already computes $w_1 x_1 + w_2 x_2$. Creating a new feature $x_1 + x_2$ just adds a column that is a *linear combination* of existing columns — it is **redundant** by definition. Ridge regression will assign it a weight that sums with the existing weights and nothing changes in the model's expressiveness.

**Always use products** ($x_1 \times x_2$) for interactions. The product creates information that *cannot* be reconstructed from $x_1$ and $x_2$ independently by any linear combination.

**Summary of interaction types**:

| Feature combination | What it captures | Add it? |
|---|---|---|
| $x_i \times x_j$ | Synergy / modulating effect | Yes — always use products |
| $x_i^2$ | Curvature (U-shape) | Yes — if you suspect non-linearity |
| $x_i + x_j$ | Additive sum | No — already captured by linear model |
| $x_i - x_j$ | Difference | Only if domain-motivated (e.g. price gap) |
| $x_i / x_j$ | Ratio | Yes — powerful for ratios (e.g. price per bedroom) |

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Domain-driven interactions for Airbnb ────────────────────────────────
# These are motivated by what a pricing analyst would reason about:

# 1. Capacity efficiency: how many guests does each bedroom serve?
#    High ratio → listing is tightly packed → may affect comfort and price
df['bedrooms_x_accommodates'] = df['bedrooms'] * df['accommodates']

# 2. Review credibility: a 4.9 from 500 reviews is more credible than 4.9 from 2 reviews
df['review_credibility'] = df['review_score'] * np.log1p(df['num_reviews'])

# 3. Location-quality: being central AND being a superhost is a double premium
df['location_x_superhost'] = (1 / (df['distance_to_centre'] + 0.1)) * df['is_superhost']

# 4. Experienced good host: long-time hosts with great reviews command more
df['experience_x_review'] = np.log1p(df['host_experience_days']) * df['review_score']

domain_interaction_cols = [
    'bedrooms_x_accommodates',
    'review_credibility',
    'location_x_superhost',
    'experience_x_review'
]

# ── Show each interaction's correlation with price ─────────────────────────
print('Correlation of domain-driven interactions with price:')
print('-'*50)
for col in domain_interaction_cols:
    r = df[col].corr(df['price'])
    print(f'  {col:35s}: r = {r:+.4f}')

# ── Visualise distributions of domain interactions ────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, col in zip(axes, domain_interaction_cols):
    ax.scatter(df[col], df['price'], alpha=0.2, s=8, color='#4472c4')
    r = df[col].corr(df['price'])
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel('Price')
    ax.set_title(f'{col}\nr = {r:.3f}', fontsize=9)

plt.suptitle('Domain-Driven Interaction Features vs. Price', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Final comparison: baseline vs baseline + top-10 MI interactions ────────
# Compare validation MAE across Ridge regularisation strengths (C values)
# Note: Ridge alpha = 1/C in the sklearn convention for some estimators,
# but Ridge() uses alpha directly. We sweep alpha = 1/C.

# Rebuild interaction matrix for this final comparison
X_all_cross_final = X_cross                               # all cross-terms computed earlier
top10_idx = mi_rank_idx[:10]                              # top-10 MI-selected interactions
X_top10   = X_all_cross_final[:, top10_idx]

X_baseline = X_numeric
X_augmented = np.hstack([X_numeric, X_top10])             # 6 + 10 = 16 features

alphas = np.logspace(-2, 4, 40)                           # 0.01 → 10 000

mae_baseline   = []
mae_augmented  = []

scaler_b = StandardScaler()
scaler_a = StandardScaler()

for alpha in alphas:
    ridge = Ridge(alpha=alpha)

    # Baseline
    pipe_b = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
    cv_b = cross_val_score(pipe_b, X_baseline, y, cv=5,
                           scoring='neg_mean_absolute_error')
    mae_baseline.append(-cv_b.mean())                     # flip sign: neg_MAE → MAE

    # Augmented
    pipe_a = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
    cv_a = cross_val_score(pipe_a, X_augmented, y, cv=5,
                           scoring='neg_mean_absolute_error')
    mae_augmented.append(-cv_a.mean())

# ── Plot MAE vs alpha ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(alphas, mae_baseline,  'b-o', linewidth=2, markersize=4, label='Baseline (6 features)')
ax.plot(alphas, mae_augmented, 'g-s', linewidth=2, markersize=4,
        label='Baseline + top-10 MI interactions')
ax.set_xscale('log')                                      # log scale: covers 6 decades
ax.set_xlabel('Ridge alpha (regularisation strength)', fontsize=12)
ax.set_ylabel('CV Mean Absolute Error (USD/night)', fontsize=12)
ax.set_title('Validation MAE vs Regularisation Strength\nInteraction Features Improve Predictions', fontsize=13)
ax.legend(fontsize=11)

# Annotate the best point for each curve
best_b_idx = int(np.argmin(mae_baseline))
best_a_idx = int(np.argmin(mae_augmented))
ax.scatter([alphas[best_b_idx]], [mae_baseline[best_b_idx]],
           color='blue', s=100, zorder=5)
ax.scatter([alphas[best_a_idx]], [mae_augmented[best_a_idx]],
           color='green', s=100, zorder=5)

plt.tight_layout()
plt.show()

print(f'Best baseline MAE   : {min(mae_baseline):.2f} USD/night')
print(f'Best augmented MAE  : {min(mae_augmented):.2f} USD/night')
print(f'Improvement         : {min(mae_baseline) - min(mae_augmented):.2f} USD/night')

## Self-Check Questions

---

**Question 1.** You have a dataset with 50 features. If you generate all degree-2 polynomial features (including squared terms and cross-products), how many total features will you have? Is it safe to fit an unregularised linear regression on 2 000 training samples with these features?

<details><summary>Show answer</summary>

Total features = original $p$ + squared $p$ + cross-terms $p(p-1)/2$ = $50 + 50 + 1225 = 1325$.

Alternatively using the formula: $p + p(p+1)/2 = 50 + 50 \times 51 / 2 = 50 + 1275 = 1325$.

With 2 000 samples and 1 325 features the ratio is about 1.5 samples per feature — far below the safe threshold of 10–50. An **unregularised** linear regression would massively overfit (possibly even be underdetermined depending on the rank of the feature matrix). You must use Ridge, Lasso, or Elastic Net regularisation, and should consider MI-based screening to reduce the feature count first.

</details>

---

**Question 2.** A colleague suggests adding the feature `bedrooms + bathrooms` to the model because "total rooms might matter". Will this improve a Ridge regression model? What would actually help?

<details><summary>Show answer</summary>

No. `bedrooms + bathrooms` is a *linear combination* of two features that are already in the model. Ridge regression (and any linear model) can already represent this relationship via $w_1 \cdot \text{bedrooms} + w_2 \cdot \text{bathrooms}$ — setting $w_1 = w_2 = 1$ reproduces the sum. Adding a redundant linear combination does not increase the model's expressiveness; it only introduces collinearity.

What would actually help: a **product** like `bedrooms × bathrooms` (captures the synergy between more rooms of both types) or a **ratio** like `bathrooms / bedrooms` (captures bathroom density, which affects comfort and price independently of total size).

</details>

---

**Question 3.** You run MI screening and find that `review_score × distance_to_centre` is in the top-5 interactions, but `bedrooms × bathrooms` is in the bottom half. Give an intuitive explanation of why this might be the case in a real Airbnb pricing context.

<details><summary>Show answer</summary>

`review_score × distance_to_centre` has a high MI because location and quality interact in a pricing-relevant way: a highly-rated listing in the city centre is worth *much* more than the sum of "high rating" and "central location" individually — the combination is what premium guests specifically seek and pay for. This synergy is real and non-linear.

`bedrooms × bathrooms` has low MI because the two features are already highly collinear (more bedrooms almost always means more bathrooms). Their product is approximately proportional to `bedrooms²`, which adds curvature but relatively little new information given that the linear terms of both features are already in the model. The marginal gain from the interaction is small because both features are capturing the same underlying dimension (listing size).

</details>

## Key Takeaways

1. **Linear models cannot learn multiplicative relationships** without explicit interaction features. If the true signal involves $x_1 \times x_2$, you must add that product as a column.

2. **Interactions explode quickly**. $p=50$ features → 1225 pairwise products. Always use screening (MI, domain knowledge, or regularisation) before adding all interactions.

3. **Use mutual information, not Pearson correlation**, to rank interactions. MI catches non-linear dependencies that correlation misses.

4. **Domain knowledge beats brute force**. Manually constructed interactions like `review_score × log(num_reviews)` are often more useful than top-ranked MI interactions from a fully automated sweep.

5. **Always regularise** when adding interaction features. Ridge or Lasso protects against overfitting when the feature space expands.

6. **Only products add new information** to a linear model. Sums and differences of existing features are already represented by the linear combination of their individual weights.

---

**Next up — Notebook 3: Encoding Categorical Features.** We will tackle the most common feature engineering challenge in real-world tabular data: turning string categories into numbers that models can actually use.